In [12]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [2]:
# Read training data
train_df = pd.read_csv('../data/train.csv')

In [4]:
# Create dataset
drop_columns = ['color', 'car_name', 'map_code', 'assists', 'mvp', 'demos_inflicted', 'demos_taken']

match_df = train_df.drop(columns=drop_columns).groupby(['match_id', 'rank']).mean().reset_index()

In [5]:
# Set features and target
X = match_df.drop(columns=['match_id', 'rank'])
y = match_df['rank']

In [13]:
le = LabelEncoder()
y = le.fit_transform(y)
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [30]:
# Split train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('imputer', SimpleImputer(), ['possession_time', 'time_in_side', 'goals_against_while_last_defender'])
    ],
    remainder='passthrough'
)

In [35]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(
            activation='tanh',
            hidden_layer_sizes=(64,64,64),
            early_stopping=True,
            n_iter_no_change=2,
            validation_fraction=0.2
        ))
    ]
).fit(X_train, y_train)

In [36]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

In [37]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       0.50      0.43      0.46       590
           1       0.74      0.69      0.71      4702
           2       0.51      0.59      0.55      5533
           3       0.55      0.64      0.59      5001
           4       0.54      0.46      0.50      5998
           5       0.61      0.48      0.54      2272

    accuracy                           0.57     24096
   macro avg       0.57      0.55      0.56     24096
weighted avg       0.58      0.57      0.57     24096

              precision    recall  f1-score   support

           0       0.50      0.44      0.47       147
           1       0.70      0.64      0.67      1176
           2       0.48      0.57      0.52      1383
           3       0.53      0.61      0.57      1251
           4       0.52      0.45      0.48      1500
           5       0.60      0.44      0.51       568

    accuracy                           0.55      6025
   macro avg       0.55

In [38]:
test_df = pd.read_csv('../data/test.csv')

In [39]:
match_test_df = test_df.drop(columns=drop_columns).groupby(['match_id']).mean().reset_index()

In [40]:
y_pred = pipe.predict(match_test_df[X.columns])

In [41]:
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [42]:
converter = { 0: 1, 5: 2, 3: 3, 4: 4, 2: 5, 1: 6 }

y_pred = pd.Series(y_pred).map(converter)

In [43]:
submission = pd.concat([match_test_df['match_id'], y_pred], axis = 1).rename(columns = {0: 'rank'})

In [44]:
submission.to_csv('../data/mlpclassification_mean_important_features_3-64_tanh_2.csv', index=False)

In [45]:
submission.shape

(2500, 2)